# Landing — trust prices from Yahoo

Source: Yahoo Finance via `yfinance`. This is the **price source for the trusts**; the
price CSV is kept only as an archive of the trusts Yahoo has erased.

The universe comes from `data/uk_investment_trusts.csv`, so it is defined in one place.
Each ticker is requested with a `.L` suffix for the London Stock Exchange.

Requested with `auto_adjust=False, actions=True`, which returns `Close`, `Adj_Close`,
`Dividends` and `Stock_Splits` rather than one pre-adjusted price. Every column returned
is landed; Silver decides which the analysis uses.

Expected: **100 of 118 symbols return data**, 35,121 rows.

In [0]:
%pip install yfinance

In [0]:
dbutils.library.restartPython()

In [0]:
import os
from datetime import datetime, timezone

import pandas as pd
import yfinance as yf

CATALOG = "`index-vs-trust-pipeline`"
PRICES_TABLE = f"{CATALOG}.landing.trust_prices_yf_raw"
LOG_TABLE = f"{CATALOG}.landing.yf_pull_log_raw"
ASSET_CLASS = "trust"

# Matches the declared column order of landing.yf_pull_log_raw.
LOG_SCHEMA = (
    "asset_class string, source_ticker string, requested_symbol string, status string, "
    "row_count int, currency string, first_bar string, last_bar string, message string, "
    "pulled_at timestamp"
)

# Notebook runs from its own folder in a Git folder, so the repo root is two levels up.
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
CSV_PATH = os.path.join(REPO_ROOT, "data", "uk_investment_trusts.csv")

In [0]:
meta = pd.read_csv(CSV_PATH, dtype=str, keep_default_na=False)

# Two rows carry a blank ticker, which is not a symbol that can be requested.
tickers = sorted({t.strip() for t in meta["ticker"] if t.strip()})

print(f"{len(tickers)} tickers to request")

In [0]:
frames = []
log_rows = []
pulled_at = datetime.now(timezone.utc)

for ticker in tickers:
    # Yahoo lists LSE securities under a .L suffix; the bare ticker resolves to nothing.
    symbol = f"{ticker}.L"

    try:
        yf_ticker = yf.Ticker(symbol)
        hist = yf_ticker.history(
            period="max", interval="1mo", auto_adjust=False, actions=True
        )
    except Exception as exc:
        log_rows.append(
            (ASSET_CLASS, ticker, symbol, "ERROR", 0, None, None, None, str(exc)[:500], pulled_at)
        )
        print(f"{symbol:10} ERROR   {exc}")
        continue

    # Yahoo returns an empty frame for delisted symbols; log the gap and skip.
    if hist.empty:
        log_rows.append(
            (ASSET_CLASS, ticker, symbol, "NODATA", 0, None, None, None, "no rows returned", pulled_at)
        )
        print(f"{symbol:10} NODATA")
        continue

    # yfinance puts the bar date in the row index, not a column.
    hist = hist.reset_index()

    # Delta rejects spaces in column names, so "Adj Close" cannot be stored as it stands.
    hist.columns = [c.replace(" ", "_") for c in hist.columns]

    # Spark cannot store a timezone-aware pandas timestamp, and yfinance omits the
    # timezone on some responses, where stripping it would raise.
    if hist["Date"].dt.tz is not None:
        hist["Date"] = hist["Date"].dt.tz_localize(None)

    # The response does not name the symbol it describes.
    hist.insert(0, "source_ticker", ticker)
    hist.insert(1, "symbol", symbol)

    # LSE trusts quote in GBp, USD or EUR and the price rows do not carry the unit.
    try:
        currency = yf_ticker.fast_info["currency"]
    except Exception:
        currency = None

    first_bar = str(hist["Date"].min().date())
    last_bar = str(hist["Date"].max().date())

    frames.append(hist)
    log_rows.append(
        (ASSET_CLASS, ticker, symbol, "OK", len(hist), currency, first_bar, last_bar, None, pulled_at)
    )
    print(f"{symbol:10} OK     {len(hist):4} bars  {first_bar} to {last_bar}  {currency}")

print(f"\n{len(frames)} of {len(tickers)} symbols returned data")

In [0]:
# A total failure would otherwise surface as an obscure pandas error on an empty list.
if not frames:
    raise RuntimeError("Yahoo returned no rows for any symbol -- check connectivity")

# This table is written with OVERWRITE, so a degraded pull would replace good data with
# less of it. On a manual run someone notices; on a schedule nobody is watching.
MIN_SYMBOLS = 90   # 100 of 118 return data today; 90 leaves room for Yahoo to lose a few
if len(frames) < MIN_SYMBOLS:
    raise RuntimeError(
        f"Only {len(frames)} of {len(tickers)} symbols returned data, below the floor "
        f"of {MIN_SYMBOLS}. Refusing to overwrite good data."
    )

prices = pd.concat(frames, ignore_index=True)

# Capital_Gains is returned for some instruments and not others, so the column set can
# shift between runs; overwriteSchema lets the table follow it.
(
    spark.createDataFrame(prices)
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(PRICES_TABLE)
)

print(f"wrote {len(prices)} rows to {PRICES_TABLE}")

In [0]:
# Delete then append this notebook's slice, so the index pull's rows survive and a
# re-run cannot double-count.
spark.sql(f"DELETE FROM {LOG_TABLE} WHERE asset_class = '{ASSET_CLASS}'")

(
    spark.createDataFrame(log_rows, LOG_SCHEMA)
    .write.format("delta")
    .mode("append")
    .saveAsTable(LOG_TABLE)
)

print(f"logged {len(log_rows)} symbols to {LOG_TABLE}")

## Verification

In [0]:
%sql
SELECT status, COUNT(*) AS symbols
FROM `index-vs-trust-pipeline`.landing.yf_pull_log_raw
WHERE asset_class = 'trust'
GROUP BY status
ORDER BY status;

Expect **OK 100, NODATA 18**, totalling the 118 requested.

The 18: `ABR ACI ADD AIS APAX ASIT BCPT CDI CSH ECWO FJV HET PLI PRSR SCIN SSON THRB
UKCM`. Sixteen get `No data found, symbol may be delisted`; `APAX` and `HET` return an
empty frame instead, which lands them in the same bucket.

Two of the 18, `BCPT` and `CSH`, have real history in the price CSV, which is why that
archive is still landed.

In [0]:
%sql
SELECT COUNT(*)              AS row_count,
       COUNT(DISTINCT symbol) AS symbols,
       MIN(`Date`)            AS first_bar,
       MAX(`Date`)            AS last_bar
FROM `index-vs-trust-pipeline`.landing.trust_prices_yf_raw;

Expect **100 symbols** and about **35,121 rows**. The count grows by roughly 100 a
month, so it is approximate; the symbol count is exact.

In [0]:
%sql
-- Dividends are the reason for this source. Prove they landed.
SELECT COUNT(DISTINCT symbol) AS symbols_paying,
       COUNT(*)               AS dividend_rows
FROM `index-vs-trust-pipeline`.landing.trust_prices_yf_raw
WHERE Dividends > 0;

Expect **90 symbols** paying dividends out of the 100.

In [0]:
%sql
-- Full history, not a window: AAS predates the CSV's 2011 floor by sixteen years.
SELECT MIN(`Date`) AS first_bar,
       MAX(`Date`) AS last_bar,
       COUNT(*)    AS bars
FROM `index-vs-trust-pipeline`.landing.trust_prices_yf_raw
WHERE symbol = 'AAS.L';

Expect a first bar of **1995-10** and **372 bars**. If this starts in 2011 the pull
has been windowed somewhere.

In [0]:
%sql
SELECT currency, COUNT(*) AS symbols
FROM `index-vs-trust-pipeline`.landing.yf_pull_log_raw
WHERE asset_class = 'trust' AND status = 'OK'
GROUP BY currency
ORDER BY symbols DESC;

Expect **96 GBp, 3 USD, 1 EUR**. Landing records the mix; normalising it is Silver's job.